# PicoCal - Spacetime transformer (notebook 16)

Space-only architectures plateau at sigma_eff ~0.065 on min-bias (nb13/nb15) - a **pileup floor** set by not being able to tell photon cells from background cells. The physical variable that tells them apart is **timing**, and we have never used it. PicoCal's fast timing (~10-20 ps) exists exactly for pileup mitigation - it time-separates the primary pp collisions (LHCb PicoCal, JINST 21 (2026) C03006; arXiv:2203.07286). This is the **spacetime** in the project title.

We add per-cell timing on top of the nb15 ParT architecture (energy-weighted EFN pooling + residual target + pairwise bias U) and run an **ablation ladder**:
1. **space** - 12 features, no timing (reproduces nb15 PairT)
2. **+time tokens** - add per-cell `dt_front, dt_back, has_valid_time` (15 features)
3. **+dt in U (spacetime)** - also feed `|dt_i - dt_j|` into the pairwise attention bias

Reported **per adaptive energy bin**: timing should help most at low energy, where pileup contamination is worst. Honest target: approach the clean floor (~0.036) overall, chase 0.02 in the high-energy bins - not one aggregate number.

In [1]:
import sys, copy, pickle, time
from pathlib import Path
import numpy as np
import pandas as pd
import awkward as ak
import uproot
import torch
import torch.nn as nn
from sklearn.ensemble import HistGradientBoostingRegressor

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from run_experiments import derive_geom, select_knn, split, resolution, CELL_KEYS, PITCH, EPS

SEEDS = 3
cfg = {"d": 96, "nhead": 4, "layers": 3, "dropout": 0.1, "lr": 3e-4, "wd": 1e-4,
       "batch": 256, "epochs": 150, "patience": 25, "pair_hidden": 32}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TSENT = 1e6   # |t| >= TSENT is the no-valid-time sentinel
{"device": DEVICE}

{'device': 'cuda'}

## Spacetime tokenizer
Reuses the committed `derive_geom`/`select_knn`. Base 7 continuous features are identical to the nb13/nb15 tokens; we append `dt_front, dt_back` (time relative to the seed cell, 0 where invalid) and a `has_valid_time` flag. Sentinel times (~-6.8e37, ~69% of cells) are masked to 0. `tok12` (no timing) is kept alongside `tok15` so the space-only rung is an exact control.

In [2]:
TKEYS = CELL_KEYS + ["cell_times_front", "cell_times_back"]
AUX = ["sig_flux_prod_vertex_z", "sig_flux_eTot", "total_energy", "x_cluster", "y_cluster"]

def build_st(files, selector, vertex_max=100.0):
    O = {k: [] for k in ["tok12", "tok15", "R", "agg", "total_energy", "y", "Etrue", "region"]}
    for path in files:
        with uproot.open(path) as f:
            a = f["clusters_matched"].arrays(TKEYS + AUX, library="ak")
        vz = ak.to_numpy(a["sig_flux_prod_vertex_z"]).astype(float)
        for i in np.flatnonzero(vz < vertex_max):
            c = {k: np.asarray(ak.to_numpy(a[k][i])).astype(float) for k in TKEYS}
            sel = selector(c)
            cw = {k: v[sel] for k, v in c.items()}
            e = cw["energy"]
            if len(e) == 0:
                continue
            pitch, mod, rx, ry, rdr, seed = derive_geom(cw)
            fr = cw["cell_energies_front"]; bk = cw["cell_energies_back"]
            tf = cw["cell_times_front"]; tb = cw["cell_times_back"]
            vf = np.abs(tf) < TSENT; vb = np.abs(tb) < TSENT
            t0f = tf[seed] if vf[seed] else (np.median(tf[vf]) if vf.any() else 0.0)
            t0b = tb[seed] if vb[seed] else (np.median(tb[vb]) if vb.any() else 0.0)
            dtf = np.where(vf, tf - t0f, 0.0)
            dtb = np.where(vb, tb - t0b, 0.0)
            hasv = (vf | vb).astype(float)
            cont7 = np.stack([np.log1p(np.clip(e, 0, None)), np.log1p(np.clip(fr, 0, None)),
                              np.log1p(np.clip(bk, 0, None)), rx / pitch, ry / pitch,
                              rdr / pitch, np.log(pitch)], 1)
            oh = np.zeros((len(e), len(PITCH))); oh[np.arange(len(e)), mod] = 1.0
            tok12 = np.concatenate([cont7, oh], 1).astype(np.float32)
            tok15 = np.concatenate([cont7, dtf[:, None], dtb[:, None], hasv[:, None], oh], 1).astype(np.float32)
            O["tok12"].append(tok12); O["tok15"].append(tok15)
            O["R"].append(np.stack([rx / pitch, ry / pitch, np.log1p(np.clip(e, 0, None)), dtf, hasv], 1).astype(np.float32))
            sumE = float(e.sum()); seedE = float(e[seed]); region = int(mod[seed])
            lat = float(np.sqrt((e * rdr ** 2).sum() / (sumE + EPS)))
            fb = float(fr.sum() / (bk.sum() + EPS))
            O["agg"].append([np.log1p(sumE), fb, len(e), np.log1p(seedE), lat, region])
            O["total_energy"].append(float(a["total_energy"][i]))
            et = float(a["sig_flux_eTot"][i])
            O["y"].append(np.log(max(et, 1e-3))); O["Etrue"].append(et); O["region"].append(region)
    for k in ["agg", "total_energy", "y", "Etrue", "region"]:
        O[k] = np.array(O[k])
    return O

cache = repo / "data" / "cache" / "minbias_spacetime_knn25.pkl"
if cache.exists():
    with open(cache, "rb") as f:
        D = pickle.load(f)
    print("loaded cache", cache.name)
else:
    files = sorted((repo / "data" / "minimum_bias").glob("matched_*.root"))
    t0 = time.time(); D = build_st(files, lambda c: select_knn(c, 25))
    cache.parent.mkdir(parents=True, exist_ok=True)
    with open(cache, "wb") as f:
        pickle.dump(D, f, protocol=4)
    print(f"built + cached {len(D['y'])} clusters in {time.time()-t0:.0f}s")
{"clusters": int(len(D["y"])), "tok15_dim": int(D["tok15"][0].shape[1])}

loaded cache minbias_spacetime_knn25.pkl


{'clusters': 89797, 'tok15_dim': 15}

In [3]:
import plotly.graph_objects as go
# timing diagnostic: dt of the seed (signal) cell vs other valid cells
seed_dt, other_dt = [], []
for R in D["R"][:5000]:
    v = R[:, 4] > 0.5
    if v.sum() >= 2:
        seed_dt.append(0.0)
        other_dt.extend(R[1:, 3][v[1:]].tolist())
fig = go.Figure()
fig.add_trace(go.Histogram(x=np.clip(other_dt, -20, 20), nbinsx=80, name="other valid cells (dt to seed)",
                           marker_color="#e45756"))
fig.update_layout(template="plotly_white", height=360, barmode="overlay",
                  title="Per-cell time relative to seed (valid cells) - in-time peak vs out-of-time pileup tail",
                  xaxis_title="dt = t_cell - t_seed", yaxis_title="cells")
fig.show()
{"valid_cell_fraction": round(float(np.mean([ (R[:,4]>0.5).mean() for R in D["R"][:5000] ])), 3)}

{'valid_cell_fraction': 0.928}

In [4]:
y = D["y"]; Et = D["Etrue"]; region = D["region"]; agg = D["agg"]
keep = np.flatnonzero((Et >= 1.0) & (Et <= 100.0))
ktr, kva, kte = (keep[s] for s in split(len(keep)))

G = np.stack([agg[:, 0], agg[:, 3], np.log(agg[:, 2] + 1.0), agg[:, 1], agg[:, 4]], 1).astype(np.float32)
n_global = G.shape[1]
la, lb = np.polyfit(agg[ktr, 0], y[ktr], 1)
base_all = (la * agg[:, 0] + lb).astype(np.float32)
gb = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(agg[ktr], y[ktr])
BDT = float(resolution(np.exp(gb.predict(agg[kte])), Et[kte])["sigma_eff"])
SUMcal = float(resolution(np.exp(la * agg[kte, 0] + lb), Et[kte])["sigma_eff"])

N = len(y); maxL = max(t.shape[0] for t in D["tok15"])

def pad_stack(toks, F):
    A = np.zeros((N, maxL, F), np.float32); M = np.zeros((N, maxL), np.bool_)
    for i, t in enumerate(toks):
        L = t.shape[0]; A[i, :L] = t; M[i, :L] = True
    return A, M

X15, Mall = pad_stack(D["tok15"], D["tok15"][0].shape[1])
X12, _ = pad_stack(D["tok12"], D["tok12"][0].shape[1])
Rpad, _ = pad_stack(D["R"], D["R"][0].shape[1])
Wall = np.zeros((N, maxL), np.float32)
for i, t in enumerate(D["tok15"]):
    L = t.shape[0]; ee = np.expm1(np.clip(t[:, 0], 0, None)); Wall[i, :L] = ee / (ee.sum() + 1e-9)

def stdz(A, ncont, tridx):
    cont = A[tridx][:, :, :ncont].reshape(-1, ncont)
    mask = Mall[tridx].reshape(-1)
    cont = cont[mask]
    mean = cont.mean(0); std = cont.std(0) + EPS
    A = A.copy(); A[:, :, :ncont] = (A[:, :, :ncont] - mean) / std; A[~Mall] = 0.0
    return A

X15 = stdz(X15, 9, ktr)
X12 = stdz(X12, 7, ktr)
gmean = G[ktr].mean(0); gstd = G[ktr].std(0) + EPS
Gall = ((G - gmean) / gstd).astype(np.float32)

T = lambda z: torch.from_numpy(z).to(DEVICE)
X15t, X12t, Mt, Wt, Rt = T(X15), T(X12), T(Mall), T(Wall), T(Rpad)
Gt = T(Gall); Bt = T(base_all).unsqueeze(1); Yt = T(y.astype(np.float32)).unsqueeze(1)
{"BDT": round(BDT, 4), "sum_calib": round(SUMcal, 4), "n_train": int(len(ktr)),
 "n_test": int(len(kte)), "maxL": int(maxL)}

{'BDT': 0.1253,
 'sum_calib': 0.1837,
 'n_train': 57056,
 'n_test': 12227,
 'maxL': 25}

In [5]:
def pair_features(R, use_time):
    rx, ry, le = R[..., 0], R[..., 1], R[..., 2]
    dx = rx.unsqueeze(2) - rx.unsqueeze(1)
    dy = ry.unsqueeze(2) - ry.unsqueeze(1)
    dR = torch.sqrt(dx * dx + dy * dy + 1e-6)
    esum = le.unsqueeze(2) + le.unsqueeze(1)
    emin = torch.minimum(le.unsqueeze(2), le.unsqueeze(1))
    feats = [dx, dy, dR, esum, emin]
    if use_time:
        dt = R[..., 3]; hv = R[..., 4]
        dtij = (dt.unsqueeze(2) - dt.unsqueeze(1)).abs()
        both = hv.unsqueeze(2) * hv.unsqueeze(1)
        feats += [dtij * both, both]
    return torch.stack(feats, -1)


class PairEmbed(nn.Module):
    def __init__(self, n_in, nhead, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, hidden), nn.GELU(),
                                 nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, nhead))
    def forward(self, pf):
        return self.net(pf).permute(0, 3, 1, 2).contiguous()


class PMHA(nn.Module):
    def __init__(self, d, nhead, dropout):
        super().__init__()
        self.h = nhead; self.dh = d // nhead
        self.q = nn.Linear(d, d); self.k = nn.Linear(d, d); self.v = nn.Linear(d, d); self.o = nn.Linear(d, d)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, U, kv):
        B, L, d = x.shape
        q = self.q(x).view(B, L, self.h, self.dh).transpose(1, 2)
        k = self.k(x).view(B, L, self.h, self.dh).transpose(1, 2)
        v = self.v(x).view(B, L, self.h, self.dh).transpose(1, 2)
        s = (q @ k.transpose(-2, -1)) / (self.dh ** 0.5)
        if U is not None:
            s = s + U
        s = s.masked_fill((~kv).view(B, 1, 1, L), -1e9)
        a = self.drop(s.softmax(-1))
        return self.o((a @ v).transpose(1, 2).reshape(B, L, d))


class Block(nn.Module):
    def __init__(self, d, nhead, dropout):
        super().__init__()
        self.n1 = nn.LayerNorm(d); self.attn = PMHA(d, nhead, dropout); self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(dropout), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(dropout)
    def forward(self, x, U, kv):
        x = x + self.drop(self.attn(self.n1(x), U, kv))
        x = x + self.drop(self.ff(self.n2(x)))
        return x


class PairT(nn.Module):
    def __init__(self, in_dim, use_time_pair):
        super().__init__()
        d = cfg["d"]; self.use_time_pair = use_time_pair
        n_pair = 7 if use_time_pair else 5
        self.embed = nn.Linear(in_dim, d)
        self.pair = PairEmbed(n_pair, cfg["nhead"], cfg["pair_hidden"])
        self.blocks = nn.ModuleList([Block(d, cfg["nhead"], cfg["dropout"]) for _ in range(cfg["layers"])])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + n_global, d), nn.ReLU(),
                                  nn.Dropout(cfg["dropout"]), nn.Linear(d, 1))
    def forward(self, x, m, w, g, base, R):
        U = self.pair(pair_features(R, self.use_time_pair))
        h = self.embed(x)
        for blk in self.blocks:
            h = blk(h, U, m)
        p = self.norm((h * w.unsqueeze(-1)).sum(1))
        return base + self.head(torch.cat([p, g], 1))

In [6]:
def train_eval(Xt, in_dim, use_time_pair, seed):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = PairT(in_dim, use_time_pair).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])

    def batches(idx, bs, sh):
        idx = np.asarray(idx)
        if sh:
            idx = rng.permutation(idx)
        for j in range(0, len(idx), bs):
            b = torch.from_numpy(idx[j:j + bs]).to(DEVICE)
            yield Xt[b], Mt[b], Wt[b], Gt[b], Bt[b], Rt[b], Yt[b]

    def run(idx):
        out = []
        with torch.no_grad():
            for X, m, w, g, base, R, _ in batches(idx, 512, False):
                out.append(model(X, m, w, g, base, R).cpu().numpy().ravel())
        return np.concatenate(out)

    def vloss():
        model.eval(); s = 0.0; c = 0
        with torch.no_grad():
            for X, m, w, g, base, R, yb in batches(kva, 512, False):
                s += nn.functional.mse_loss(model(X, m, w, g, base, R), yb).item(); c += 1
        return s / max(c, 1)

    best = 1e9; bstate = None; wait = 0
    for ep in range(cfg["epochs"]):
        model.train()
        for X, m, w, g, base, R, yb in batches(ktr, cfg["batch"], True):
            opt.zero_grad()
            nn.functional.mse_loss(model(X, m, w, g, base, R), yb).backward(); opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4:
            best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
            if wait >= cfg["patience"]:
                break
    model.load_state_dict(bstate); model.eval()
    a, b = np.polyfit(run(kva), y[kva], 1)
    pe = np.exp(a * run(kte) + b)
    return float(resolution(pe, Et[kte])["sigma_eff"]), pe

LADDER = [("space (no timing)", X12t, 12, False),
          ("+time tokens", X15t, 15, False),
          ("+dt in U (spacetime)", X15t, 15, True)]
res16 = {}; best_pe = {}; rows = []
t0 = time.time()
for name, Xt, ind, utp in LADDER:
    vals = []; pe_last = None
    for s in range(SEEDS):
        sig, pe = train_eval(Xt, ind, utp, s); vals.append(sig); pe_last = pe
        print(f"  {name} seed {s}: {sig:.4f}", flush=True)
    best_pe[name] = pe_last
    mean, std = float(np.mean(vals)), float(np.std(vals))
    res16[name] = vals
    rows.append({"config": name, "sigma_eff": round(mean, 4), "std": round(std, 4),
                 "BDT": round(BDT, 4), "beats_BDT": mean < BDT})
    print(f"{name}: {mean:.4f} +/- {std:.4f}", flush=True)
print(f"elapsed {time.time()-t0:.0f}s")
summary16 = pd.DataFrame(rows)
summary16

  space (no timing) seed 0: 0.0687


  space (no timing) seed 1: 0.0651


  space (no timing) seed 2: 0.0658


space (no timing): 0.0665 +/- 0.0016


  +time tokens seed 0: 0.0565


...

  +dt in U (spacetime) seed 1: 0.0590


  +dt in U (spacetime) seed 2: 0.0577


+dt in U (spacetime): 0.0589 +/- 0.0010


elapsed 6670s


,config,sigma_eff,std,BDT,beats_BDT
0,space (no timing),0.0665,0.0016,0.1253,True
1,+time tokens,0.0564,0.0007,0.1253,True
2,+dt in U (spacetime),0.0589,0.0010,0.1253,True


In [7]:
d = summary16
fig = go.Figure(go.Bar(x=d["config"], y=d["sigma_eff"], error_y=dict(type="data", array=d["std"]),
                       marker_color=["#8c8c8c", "#4c78a8", "#1f77b4"],
                       text=[f"{v:.4f}" for v in d["sigma_eff"]], textposition="outside"))
fig.add_hline(y=BDT, line_dash="dash", line_color="crimson", annotation_text=f"fair BDT {BDT:.4f}")
fig.add_hline(y=0.02, line_dash="dot", line_color="green", annotation_text="goal 0.02")
fig.update_layout(template="plotly_white", height=460, yaxis_title="sigma_eff (min-bias)",
                  title="Timing ablation ladder: does time break the space-only floor?")
fig.show()

In [8]:
def sig_bins(pe, tr, n=8):
    edges = np.quantile(tr, np.linspace(0, 1, n + 1)); cx, sy = [], []
    for i in range(n):
        hi = edges[i + 1] + (1e-6 if i == n - 1 else 0.0)
        mk = (tr >= edges[i]) & (tr < hi)
        if mk.sum() >= 20:
            cx.append(float(np.median(tr[mk]))); sy.append(float(resolution(pe[mk], tr[mk])["sigma_eff"]))
    return np.array(cx), np.array(sy)

fig = go.Figure()
for name, col in [("space (no timing)", "#8c8c8c"), ("+dt in U (spacetime)", "#1f77b4")]:
    cx, sy = sig_bins(best_pe[name], Et[kte])
    fig.add_trace(go.Scatter(x=cx, y=sy, mode="lines+markers", name=name, line=dict(color=col)))
fig.add_hline(y=0.02, line_dash="dot", line_color="green", annotation_text="0.02")
fig.update_layout(template="plotly_white", height=430, xaxis_title="E_true [GeV] (bin median)",
                  yaxis_title="sigma_eff", title="Resolution vs energy (adaptive bins): space vs spacetime")
fig.show()

## Read-out
- **Did timing break the floor?** Compare rung 1 (space) to rungs 2-3. A real drop means per-cell timing carries pileup-rejection information the energy+geometry tokens do not - the spacetime thesis.
- **Where?** The per-energy plot shows if the gain is concentrated at low energy (most pileup contamination) as expected, and whether any high-energy bin reaches ~0.02.
- **Honest floor:** if timing pulls the aggregate toward the clean ~0.036 but 0.02 only appears in the top bins, that is the correct result - and it directly demonstrates *why the detector has timing*. Caveat: sim timing may be idealized vs the real ~20 ps; treat gains as an upper bound.